# 05 – Kreuzvalidierung ohne Data Leakage

**Lernziele:** Folds verstehen, passende Splitter wählen und Vorverarbeitung innerhalb jedes Folds lernen. Dauer: ca. 35 Minuten.

Hier dienen einfache fertige Modelle als Messinstrument. Ihre Funktionsweise folgt in Set 5. Für diese Lektion genügen `fit`, `predict` und ein Bewertungsscore.

CV teilt die Entwicklungsdaten mehrfach in Training und Validierung. Jeder Fold lernt seine eigene Pipeline. Der finale Test bleibt außerhalb der CV. Bei unabhängiger Regression passt `KFold`, bei unabhängiger Klassifikation häufig `StratifiedKFold`, bei neuen Gruppen `GroupKFold`, bei Zukunftsprognosen `TimeSeriesSplit`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, GroupKFold, GroupShuffleSplit, TimeSeriesSplit, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge


In [ ]:
X, y = make_classification(n_samples=900, n_features=8, weights=[0.85, 0.15], random_state=42)
X_entw, X_test, y_entw, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(pipeline, X_entw, y_entw, cv=cv,
                        scoring={'bal_acc': 'balanced_accuracy', 'ap': 'average_precision'}, error_score='raise')
display(pd.DataFrame({k: v for k, v in scores.items() if k.startswith('test_')}))
print('Mittlere Balanced Accuracy:', scores['test_bal_acc'].mean().round(3))
print('Streuung über Folds:', scores['test_bal_acc'].std().round(3))


`test_...` im CV-Ergebnis bezeichnet die **Validierungsfolds**, nicht den finalen Testsatz. Balanced Accuracy mittelt Trefferquoten je Klasse; Average Precision bewertet das Ranking positiver Fälle. Fold-Streuung ist kein direktes Konfidenzintervall, denn Trainingsmengen überlappen.

**Leakage-Falle:** Vor der CV auf allen Entwicklungsdaten zu skalieren lernt bereits aus späteren Validierungsfolds. Die Pipeline verhindert das. Dasselbe gilt für Imputation, PCA, Merkmalsauswahl und SMOTE.

## 2. Folds sichtbar machen


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
for fold, (tr, va) in enumerate(cv.split(X_entw, y_entw)):
    ax.scatter(tr, np.full(len(tr), fold), s=5, c='steelblue')
    ax.scatter(va, np.full(len(va), fold), s=5, c='orange')
ax.set(xlabel='Zeilenindex der Entwicklungsdaten', ylabel='Fold', title='Blau: Training, Orange: Validierung')
plt.tight_layout(); plt.show()


## 3. Gruppen-CV

Auch der äußere Holdout trennt nach Maschinen. Innere Folds trennen erneut nach Gruppen. Es müssen mindestens so viele Trainingsgruppen wie Folds existieren. Gleiche Klassenanteile sind dabei nicht garantiert; bei Klassifikation kann `StratifiedGroupKFold` helfen, soweit die Gruppenstruktur es erlaubt.


In [ ]:
rng = np.random.default_rng(42)
maschinen = np.repeat(np.arange(30), 12)
sensor = pd.DataFrame({'maschine': maschinen,
                      'vibration': np.repeat(rng.normal(0, 2, 30), 12) + rng.normal(0, 0.2, 360)})


In [ ]:
ziel = 2*sensor['vibration'].to_numpy() + rng.normal(0, 0.3, len(sensor))
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(sensor, groups=maschinen))
X_g = sensor[['vibration']].iloc[tr]
y_g, gruppen = ziel[tr], maschinen[tr]
gruppen_cv = GroupKFold(n_splits=4)
for a, b in gruppen_cv.split(X_g, y_g, groups=gruppen):
    assert set(gruppen[a]).isdisjoint(gruppen[b])
ergebnis = cross_validate(make_pipeline(StandardScaler(), Ridge()), X_g, y_g,
                         groups=gruppen, cv=gruppen_cv, scoring='neg_mean_absolute_error', error_score='raise')
print('MAE je Gruppenfold:', -ergebnis['test_score'])


## 4. Zeit-CV mit wachsendem Trainingsfenster

Die letzten 40 Stunden bleiben Test. `TimeSeriesSplit` verwendet in jedem Fold nur frühere Daten zum Training. `gap=2` lässt illustrativ zwei Zeilen frei. Die fachlich nötige Lücke folgt aus Horizont und Label-Verfügbarkeit. Gleichmäßige Zeitabstände machen die Fold-Zeiträume vergleichbar.


In [ ]:
t = np.arange(240)
y_zeit = 0.05*t + np.sin(t/8)
X_zeit = t.reshape(-1, 1)
X_frueh, y_frueh = X_zeit[:-40], y_zeit[:-40]
zeit_cv = TimeSeriesSplit(n_splits=4, test_size=25, gap=2)
grenzen = []
for fold, (a, b) in enumerate(zeit_cv.split(X_frueh)):
    assert a.max() + 2 < b.min()
    grenzen.append([fold, a.min(), a.max(), b.min(), b.max()])
display(pd.DataFrame(grenzen, columns=['Fold', 'Train ab', 'Train bis', 'Validierung ab', 'Validierung bis']))
zeit_scores = cross_validate(make_pipeline(StandardScaler(), Ridge()), X_frueh, y_frueh,
                            cv=zeit_cv, scoring='neg_mean_absolute_error', error_score='raise')
print('MAE je Zeitfold:', -zeit_scores['test_score'])


## 5. Nach allen Entscheidungen einmal testen

Die Klassifikationspipeline wird auf allen Entwicklungsdaten neu gelernt. Ein schlechter finaler Testscore rechtfertigt kein wiederholtes Tuning auf diesem Testsatz.


In [ ]:
from sklearn.metrics import balanced_accuracy_score
pipeline.fit(X_entw, y_entw)
print('Finale Balanced Accuracy:', round(balanced_accuracy_score(y_test, pipeline.predict(X_test)), 3))


**Mini-Aufgabe:** Warum ist `KFold(shuffle=True)` für Zukunftsprognosen falsch? Warum kann Stratifikation denselben Patienten auf beiden Seiten nicht rechtfertigen? Wie viele Fits brauchen drei Varianten mit 5-Fold-CV und ein finaler Fit?

Quelle: [scikit-learn: Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html). Hyperparameter-Suche wird später vertieft.
